# Exploratory Data Analysis (EDA) - JazzCash Fraud Detection

This notebook performs comprehensive exploratory data analysis on the JazzCash transaction data to understand patterns, distributions, and fraud characteristics.

## Contents
1. Setup and Data Loading
2. Basic Statistics
3. Transaction Analysis
4. Fraud Analysis
5. Channel and Type Analysis
6. Temporal Patterns
7. Amount Distributions
8. Geographic Analysis
9. Correlation Analysis

## 1. Setup and Data Loading

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql import SparkSession

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)

print("Libraries imported successfully")

In [ ]:
# Create Spark session
spark = SparkSession.builder \
    .appName("JazzCash-EDA") \
    .config("spark.driver.memory", "8g") \
    .config("spark.sql.shuffle.partitions", "20") \
    .getOrCreate()

print(f"Spark version: {spark.version}")

In [ ]:
# Load cleaned dataset
df_spark = spark.read.parquet('../data/processed/cleaned_dataset.parquet')

print(f"Total rows: {df_spark.count():,}")
print(f"Total columns: {len(df_spark.columns)}")

# Show schema
df_spark.printSchema()

In [ ]:
# Sample data for visualization (convert to Pandas)
sample_size = min(100000, df_spark.count())
df = df_spark.sample(fraction=sample_size/df_spark.count(), seed=42).toPandas()

print(f"Sample size: {len(df):,} rows")
df.head()

## 2. Basic Statistics

In [ ]:
# Basic info
df.info()

In [ ]:
# Descriptive statistics
df.describe()

In [ ]:
# Missing values analysis
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).sort_values(ascending=False)

print("Top 20 columns with missing values:")
print(missing_pct[missing_pct > 0].head(20))

## 3. Transaction Analysis

In [ ]:
# Transaction amount distribution
plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
df['trx_amt'].hist(bins=50, edgecolor='black')
plt.xlabel('Transaction Amount')
plt.ylabel('Frequency')
plt.title('Transaction Amount Distribution')

plt.subplot(1, 2, 2)
df['trx_amt'].plot(kind='box', vert=False)
plt.xlabel('Transaction Amount')
plt.title('Transaction Amount Boxplot')

plt.tight_layout()
plt.savefig('../plots/eda/amount_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"Mean amount: {df['trx_amt'].mean():.2f}")
print(f"Median amount: {df['trx_amt'].median():.2f}")
print(f"Max amount: {df['trx_amt'].max():.2f}")

## 4. Fraud Analysis

In [ ]:
# Fraud rate
if 'is_fraud' in df.columns:
    fraud_rate = df['is_fraud'].mean()
    fraud_count = df['is_fraud'].sum()
    
    print(f"Fraud Cases: {fraud_count:,}")
    print(f"Non-Fraud Cases: {(len(df) - fraud_count):,}")
    print(f"Fraud Rate: {fraud_rate:.4%}")
    
    # Pie chart
    plt.figure(figsize=(8, 8))
    df['is_fraud'].value_counts().plot(kind='pie', autopct='%1.4f%%', labels=['Non-Fraud', 'Fraud'])
    plt.title('Fraud vs Non-Fraud Transactions')
    plt.ylabel('')
    plt.show()

In [ ]:
# Fraud vs Non-Fraud amount comparison
if 'is_fraud' in df.columns:
    plt.figure(figsize=(12, 6))
    
    df[df['trx_amt'] < df['trx_amt'].quantile(0.95)].boxplot(
        column='trx_amt', by='is_fraud', figsize=(10, 6)
    )
    plt.xlabel('Is Fraud')
    plt.ylabel('Transaction Amount')
    plt.title('Transaction Amount by Fraud Status')
    plt.suptitle('')
    plt.show()
    
    print("\nAmount Statistics by Fraud Status:")
    print(df.groupby('is_fraud')['trx_amt'].describe())

## 5. Channel and Type Analysis

In [ ]:
# Transaction by channel
if 'trx_channel' in df.columns:
    plt.figure(figsize=(12, 6))
    df['trx_channel'].value_counts().plot(kind='bar')
    plt.xlabel('Channel')
    plt.ylabel('Count')
    plt.title('Transactions by Channel')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
    
    print("\nChannel Distribution:")
    print(df['trx_channel'].value_counts())

In [ ]:
# Fraud rate by channel
if 'trx_channel' in df.columns and 'is_fraud' in df.columns:
    fraud_by_channel = df.groupby('trx_channel').agg({
        'is_fraud': ['mean', 'sum', 'count']
    }).round(4)
    
    fraud_by_channel.columns = ['fraud_rate', 'fraud_count', 'total_count']
    fraud_by_channel = fraud_by_channel.sort_values('fraud_rate', ascending=False)
    
    print("Fraud Rate by Channel:")
    print(fraud_by_channel)
    
    # Plot
    fig, ax1 = plt.subplots(figsize=(12, 6))
    
    x = range(len(fraud_by_channel))
    ax1.bar(x, fraud_by_channel['fraud_rate'], alpha=0.7, label='Fraud Rate')
    ax1.set_xlabel('Channel')
    ax1.set_ylabel('Fraud Rate', color='b')
    ax1.tick_params(axis='y', labelcolor='b')
    plt.xticks(x, fraud_by_channel.index, rotation=45, ha='right')
    
    ax2 = ax1.twinx()
    ax2.plot(x, fraud_by_channel['total_count'], 'r-o', label='Total Count')
    ax2.set_ylabel('Total Count', color='r')
    ax2.tick_params(axis='y', labelcolor='r')
    
    plt.title('Fraud Rate and Count by Channel')
    fig.tight_layout()
    plt.savefig('../plots/eda/fraud_rate_by_channel.png', dpi=300, bbox_inches='tight')
    plt.show()

## 6. Temporal Patterns

In [ ]:
# Convert timestamp column
if 'trans_initiate_time' in df.columns:
    df['trans_initiate_time'] = pd.to_datetime(df['trans_initiate_time'])
    df['hour'] = df['trans_initiate_time'].dt.hour
    df['day_of_week'] = df['trans_initiate_time'].dt.dayofweek
    
    # Hourly distribution
    plt.figure(figsize=(14, 6))
    df['hour'].value_counts().sort_index().plot(kind='bar')
    plt.xlabel('Hour of Day')
    plt.ylabel('Transaction Count')
    plt.title('Transaction Distribution by Hour')
    plt.tight_layout()
    plt.show()

In [ ]:
# Fraud rate by hour
if 'hour' in df.columns and 'is_fraud' in df.columns:
    fraud_by_hour = df.groupby('hour')['is_fraud'].agg(['mean', 'count'])
    
    fig, ax1 = plt.subplots(figsize=(14, 6))
    
    ax1.plot(fraud_by_hour.index, fraud_by_hour['mean'], 'b-o', linewidth=2)
    ax1.set_xlabel('Hour of Day')
    ax1.set_ylabel('Fraud Rate', color='b')
    ax1.tick_params(axis='y', labelcolor='b')
    
    ax2 = ax1.twinx()
    ax2.bar(fraud_by_hour.index, fraud_by_hour['count'], alpha=0.3, color='gray')
    ax2.set_ylabel('Transaction Count', color='gray')
    
    plt.title('Fraud Rate by Hour of Day')
    fig.tight_layout()
    plt.show()

## 7. Correlation Analysis

In [ ]:
# Select numerical columns
numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()

# Limit to key columns for visualization
key_cols = ['trx_amt', 'fee', 'fed', 'start_balance', 'end_balance']
if 'is_fraud' in numerical_cols:
    key_cols.append('is_fraud')

key_cols = [col for col in key_cols if col in numerical_cols]

# Correlation matrix
corr = df[key_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.3f', cmap='coolwarm', center=0,
            square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Correlation Matrix - Key Features')
plt.tight_layout()
plt.savefig('../plots/eda/correlation_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Correlation with fraud
if 'is_fraud' in numerical_cols:
    fraud_corr = df[numerical_cols].corr()['is_fraud'].abs().sort_values(ascending=False)
    
    print("Top 20 Features Correlated with Fraud:")
    print(fraud_corr.head(20))
    
    # Plot
    plt.figure(figsize=(12, 8))
    fraud_corr.head(20).plot(kind='barh')
    plt.xlabel('Absolute Correlation with Fraud')
    plt.title('Top 20 Features Correlated with Fraud')
    plt.tight_layout()
    plt.show()

## 8. Summary and Key Findings

In [ ]:
print("="*80)
print("KEY FINDINGS")
print("="*80)

print(f"\n1. Dataset Size: {len(df):,} transactions (sample)")

if 'is_fraud' in df.columns:
    print(f"\n2. Fraud Statistics:")
    print(f"   - Fraud Rate: {df['is_fraud'].mean():.4%}")
    print(f"   - Fraud Cases: {df['is_fraud'].sum():,}")

print(f"\n3. Transaction Amounts:")
print(f"   - Mean: {df['trx_amt'].mean():.2f}")
print(f"   - Median: {df['trx_amt'].median():.2f}")
print(f"   - 95th Percentile: {df['trx_amt'].quantile(0.95):.2f}")

if 'trx_channel' in df.columns:
    print(f"\n4. Most Common Channel: {df['trx_channel'].mode()[0]}")

if 'hour' in df.columns:
    print(f"\n5. Peak Transaction Hour: {df['hour'].mode()[0]}")

print("\n" + "="*80)

In [ ]:
# Stop Spark session
spark.stop()
print("Spark session stopped")